# Yelp Review Emotion Classification with Hugging Face

This notebook applies the Hugging Face model `j-hartmann/emotion-english-distilroberta-base` to the Yelp review dataset stored in Google Drive.

It loads `yelp_reviews_clean.csv`, uses the `text` column as review input, generates emotion probabilities, saves a new CSV with emotion outputs, generates histograms, computes the average emotion distribution, and identifies the emotion with the highest average score.

This version keeps all seven emotion labels predicted by the model: anger 🤬, disgust 🤢, fear 😨, joy 😀, neutral 😐, sadness 😭, and surprise 😲.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Install required packages

In [ ]:
!pip install -q transformers accelerate tqdm

## 3. Import libraries

In [ ]:
import os
import gc
import json
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import pipeline
from tqdm.auto import tqdm

## 4. Set file paths

Change these paths if your files are stored in a different Drive folder.

In [ ]:
input_path = "/content/drive/MyDrive/yelp_reviews_clean.csv"

output_path = "/content/drive/MyDrive/yelp_reviews_with_hf_emotions.csv"

summary_output_path = "/content/drive/MyDrive/yelp_emotion_summary_hf.csv"

dominant_counts_output_path = "/content/drive/MyDrive/yelp_dominant_emotion_counts_hf.csv"

histogram_folder = "/content/drive/MyDrive/yelp_emotion_histograms_hf"
os.makedirs(histogram_folder, exist_ok=True)

print("Input exists:", os.path.exists(input_path))
print("Output path:", output_path)

## 5. Load the Hugging Face emotion model

This uses `j-hartmann/emotion-english-distilroberta-base`, a pretrained transformer-based emotion classifier. If Colab has a GPU available, the pipeline will use it automatically.

In [ ]:
model_name = "j-hartmann/emotion-english-distilroberta-base"

device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

emotion_classifier = pipeline(
    task="text-classification",
    model=model_name,
    top_k=None,
    truncation=True,
    device=device
)

## 6. Test the model on one sentence

In [ ]:
test_text = "I'm so annoyed that this code won't run."

test_result = emotion_classifier(test_text)
print(test_result)

## 7. Helper functions

The model returns seven emotion labels. These functions store all seven model emotions: anger, disgust, fear, joy, neutral, sadness, and surprise.


In [ ]:
target_emotions = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
]

emotion_display_names = {
    "anger": "anger 🤬",
    "disgust": "disgust 🤢",
    "fear": "fear 😨",
    "joy": "joy 😀",
    "neutral": "neutral 😐",
    "sadness": "sadness 😭",
    "surprise": "surprise 😲"
}


def normalize_pipeline_output(result):
    # For one text, result may be a list of dictionaries.
    # For batched text, each item is a list of dictionaries.
    if isinstance(result, list) and len(result) > 0:
        if isinstance(result[0], dict):
            return [result]
        return result
    return []


def extract_target_emotion_scores(single_prediction):
    # Convert one model prediction into a dictionary containing all seven emotions.
    score_dict = {emotion: np.nan for emotion in target_emotions}

    for item in single_prediction:
        label = str(item["label"]).lower()
        score = float(item["score"])

        if label in score_dict:
            score_dict[label] = score

    return score_dict


def classify_emotion_batch(texts):
    # Run batched emotion classification and return a list of dictionaries.
    cleaned_texts = []

    for text in texts:
        if pd.isna(text) or str(text).strip() == "":
            cleaned_texts.append(" ")
        else:
            cleaned_texts.append(str(text))

    predictions = emotion_classifier(
        cleaned_texts,
        batch_size=32,
        truncation=True
    )

    predictions = normalize_pipeline_output(predictions)

    emotion_scores = [
        extract_target_emotion_scores(prediction)
        for prediction in predictions
    ]

    return emotion_scores


## 8. Optional quick test on a small sample

Run this before processing the full dataset to verify that the model and file path work.

In [ ]:
sample_df = pd.read_csv(
    input_path,
    usecols=["text"],
    nrows=10
)

sample_scores = classify_emotion_batch(sample_df["text"].tolist())

sample_output = pd.concat(
    [sample_df, pd.DataFrame(sample_scores)],
    axis=1
)

sample_output["dominant_emotion"] = sample_output[target_emotions].idxmax(axis=1)

display(sample_output)

## 9. Benchmark runtime on 1,000 reviews

This estimates how long the full dataset will take on the current runtime.

In [ ]:
benchmark_size = 1000

benchmark_df = pd.read_csv(
    input_path,
    usecols=["text"],
    nrows=benchmark_size
)

start = time.time()
_ = classify_emotion_batch(benchmark_df["text"].tolist())
elapsed = time.time() - start

reviews_per_second = benchmark_size / elapsed
estimated_total_reviews = 6_970_894
estimated_hours = estimated_total_reviews / reviews_per_second / 3600

print(f"Elapsed for {benchmark_size:,} reviews: {elapsed:.2f} seconds")
print(f"Reviews per second: {reviews_per_second:.2f}")
print(f"Estimated time for {estimated_total_reviews:,} reviews: {estimated_hours:.2f} hours")

## 10. Process the full CSV in chunks



In [ ]:
chunk_size = 5_000
max_rows_to_process = None  # Example: set to 100_000 for a smaller project sample

first_chunk = True
total_rows_processed = 0

if os.path.exists(output_path):
    os.remove(output_path)

reader = pd.read_csv(
    input_path,
    chunksize=chunk_size
)

for chunk_number, chunk in enumerate(reader, start=1):

    if max_rows_to_process is not None:
        remaining_rows = max_rows_to_process - total_rows_processed
        if remaining_rows <= 0:
            break
        chunk = chunk.head(remaining_rows)

    print(f"Processing chunk {chunk_number} with {len(chunk):,} rows...")

    emotion_results = classify_emotion_batch(chunk["text"].tolist())
    emotion_df = pd.DataFrame(emotion_results)

    for emotion in target_emotions:
        chunk[emotion] = emotion_df[emotion].values

    chunk["emotion_breakdown"] = emotion_df[target_emotions].apply(
        lambda row: {emotion: row[emotion] for emotion in target_emotions},
        axis=1
    )

    chunk["emotion_breakdown"] = chunk["emotion_breakdown"].apply(json.dumps)

    chunk["dominant_emotion"] = chunk[target_emotions].idxmax(axis=1)

    chunk.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding="utf-8"
    )

    first_chunk = False
    total_rows_processed += len(chunk)

    print(
        f"Finished chunk {chunk_number}; "
        f"total rows processed: {total_rows_processed:,}"
    )

    del chunk, emotion_df, emotion_results
    gc.collect()

print("Emotion classification completed.")
print("Output saved to:", output_path)
print("Total rows processed:", f"{total_rows_processed:,}")

## 11. Load emotion columns only for analysis

This avoids loading the full output dataset into memory.

In [ ]:
emotion_data = pd.read_csv(
    output_path,
    usecols=target_emotions + ["dominant_emotion"]
)

print(emotion_data.shape)
display(emotion_data.head())

## 12. Compute the average emotion distribution

In [ ]:
average_emotions = emotion_data[target_emotions].mean().sort_values(ascending=False)

print("Average emotion distribution:")
print(average_emotions)

max_average_emotion = average_emotions.idxmax()
max_average_value = average_emotions.max()

print()
print("Emotion with highest average score:", max_average_emotion)
print("Average score:", max_average_value)

## 13. Save the average emotion summary

In [ ]:
summary_df = average_emotions.reset_index()
summary_df.columns = ["emotion", "average_score"]

summary_df.to_csv(
    summary_output_path,
    index=False
)

print("Emotion summary saved to:", summary_output_path)
display(summary_df)

## 14. Plot average emotion distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(summary_df["emotion"], summary_df["average_score"])
plt.title("Average Emotion Distribution Across Yelp Reviews")
plt.xlabel("Emotion")
plt.ylabel("Average Probability")
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.tight_layout()

average_plot_path = os.path.join(
    histogram_folder,
    "average_emotion_distribution_hf.png"
)

plt.savefig(average_plot_path, dpi=300)
plt.show()

print("Saved plot to:", average_plot_path)

## 15. Generate histograms for each emotion

In [ ]:
for emotion in target_emotions:
    plt.figure(figsize=(8, 5))

    plt.hist(
        emotion_data[emotion].dropna(),
        bins=50
    )

    plt.title(f"Distribution of {emotion.capitalize()} Scores")
    plt.xlabel(f"{emotion.capitalize()} Probability")
    plt.ylabel("Number of Reviews")
    plt.xlim(0, 1)
    plt.tight_layout()

    plot_path = os.path.join(
        histogram_folder,
        f"{emotion}_distribution_hf.png"
    )

    plt.savefig(plot_path, dpi=300)
    plt.show()

    print("Saved:", plot_path)

## 18. Inspect sample review outputs

In [ ]:
sample_output = pd.read_csv(
    output_path,
    usecols=["business_name", "stars", "sentiment", "text"] + target_emotions + ["dominant_emotion"],
    nrows=20
)

pd.set_option("display.max_colwidth", 300)
display(sample_output)